# multi agent를 이용한 고객 응대 chatbot 업그레이드

## 환경 설정

In [1]:
# 필요한 모듈, 클래스, 함수 임포트
import os
from typing import Annotated
from dotenv import load_dotenv
from pydantic import BaseModel
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END, add_messages
from langchain_chroma import Chroma
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

C:\Users\playdata2\AppData\Local\Temp\ipykernel_9892\2068573794.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


In [2]:
# .env file -> API Key 가져오기
file_path='.env'
output=load_dotenv(file_path, override=True)
print(output)

True


In [3]:
# 공통 LLM 생성
llm = ChatOpenAI(model="gpt-5-nano")

## RAG 시스템 기반 설정

### 데이터 로드 및 변환

In [4]:
# 파일 경로 설정
csv_path = "faq_manual.csv"

# CSV 파일 로드 및 변환 도구 생성 -> CSVLoader 클래스 호출
loader = CSVLoader(
    file_path=csv_path,
    encoding="utf-8",
    # 본문(page_content)에 포함할 컬럼 지정
    content_columns=["question", "answer"]
)

# load() 함수 호출 -> Document로 변환
docs = loader.load()

print(f"로드된 총 문서 수: {len(docs)}개")
print("-" * 60)
print(f"[첫 번째 Document 본문(page_content)]\n{docs[0].page_content}")
print("-" * 60)
print(f"[첫 번째 Document 메타데이터(metadata)]\n{docs[0].metadata}")

로드된 총 문서 수: 10개
------------------------------------------------------------
[첫 번째 Document 본문(page_content)]
question: 코드크래프터스 아카데미에 어떻게 가입하나요?
answer: 저희 웹사이트 우측 상단의 '회원가입' 버튼을 클릭하시고, 이메일 주소와 비밀번호만 입력하면 간단하게 가입이 완료됩니다. 이메일 인증 후 바로 모든 서비스를 이용하실 수 있습니다.
------------------------------------------------------------
[첫 번째 Document 메타데이터(metadata)]
{'source': 'faq_manual.csv', 'row': 0}


### 텍스트 임베딩 모델 생성

In [5]:
# 모델 이름 설정
model_name = "text-embedding-3-small"

# OpenAI 임베딩 모델 생성
model = OpenAIEmbeddings(
    model=model_name
)

### 백터 저장소 및 검색 도구 생성 

In [6]:
# 벡터 저장소 저장 경로 설정
chroma_path = 'chroma_db_multi'

# 이미 생성된 벡터 저장소가 있다면 불러오고, 없다면 새로 생성
if os.path.exists(chroma_path) and os.listdir(chroma_path):
    vectorstore = Chroma(
        persist_directory=chroma_path,
        embedding_function=model
    )
    print("기존 벡터 저장소를 성공적으로 불러왔습니다.")
else:
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=model,
        persist_directory=chroma_path
    )
    print("신규 벡터 저장소를 생성하고 저장했습니다.")

기존 벡터 저장소를 성공적으로 불러왔습니다.


In [7]:
# 검색 도구 생성 -> as_retriever() 함수 호출
retriever = vectorstore.as_retriever(search_kwargs={'k': 1})

In [8]:
# 검색 도구 성능 테스트
user_query = "환불은"
output = retriever.invoke(user_query)
print(f"검색 결과: \n{output}")

검색 결과: 
[Document(id='67981998-ad66-438b-a095-d948db7c9fe8', metadata={'row': 1, 'source': 'faq_manual.csv'}, page_content='question: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?\nanswer: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.')]


## 에이전트별 전용 체인 생성

In [9]:
# [질문 재구성 체인] 모호한 질문을 명확한 질문으로 변환

## system 프롬프트 생성
rephrase_system_prompt = """
주어진 채팅 기록(chat history)과 최신 사용자 질문이 있을 때,
이 질문이 채팅 기록의 맥락을 참조하고 있다면 모호한 질문(예: "그거 전액 다 받으려면 조건이 뭐야?"),
채팅 기록 없이도 이해할 수 있는 명확한 질문(예: 수강 신청한 강의를 전액 환불 받으려면 조건이 뭐야?)으로 다시 작성하세요.
질문에 답하지 말고, 오직 질문을 재구성하거나 그대로 반환하기만 하세요.
"""

## 전체 프롬프트 생성
rephrase_prompt = ChatPromptTemplate.from_messages([
    ("system", rephrase_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])
print(rephrase_prompt)

## chain 생성
rephrase_chain = rephrase_prompt | llm | StrOutputParser()

input_variables=['chat_history', 'input'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.messag

In [10]:
# [답변 생성 체인] 검색된 문서와 맥락을 바탕으로 최종 답변 작성

## system 프롬프트 생성
qa_system_prompt = """
당신은 코드크래프터스 아카데미의 친절한 고객 지원 담당자입니다.
아래의 [Context] 정보를 사용하여 사용자의 질문에 답변하세요.
정보가 없다면 '죄송하지만 매뉴얼에서 정보를 찾을 수 없습니다'라고 답하세요.

[Context]
{context}
"""

## 전체 프롬프트 생성
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])
print(qa_prompt)

## chain 생성
qa_chain = qa_prompt | llm

input_variables=['chat_history', 'context', 'input'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_

## 그래프 상태 정의

In [11]:
# pydantic 라이브러리 -> BaseModel 클래스 상속 -> 커스텀 자료형 생성
class AgentState(BaseModel):
    # 대화 내역 누적 항목
    messages: Annotated[list[BaseMessage], add_messages] = []
    # 재구성된 질문 저장
    rephrased_query: str = ""
    # 검색된 매뉴얼 내용 저장
    context: str = ""

## 그래프 노드 함수 정의

In [12]:
# 1. 질문 재구성 노드 함수 정의(비동기 처리 방식)
async def rephrase_node(state: AgentState) -> dict:
    print("에이전트 1: 질문 맥락 분석 및 재구성")
    print('-'*80)    

    # 대화 히스토리 -> 사용자 최신 질문 추출
    latest_question = state.messages[-1].content

    # 대화 히스토리 -> 사용자 최신 질문 이전 대화 내역 -> 리스트 자료형
    chat_history = state.messages[:-1]

    # 이전 대화 기록이 존재할 때만 재구성 체인 실행 
    # 비동기 처리 방식 chain 실행 -> invoke() 함수 -> await 예약어, ainvoke() 함수  
    if chat_history:
        query = await rephrase_chain.ainvoke({
            "chat_history": chat_history, 
            "input": latest_question
        })
    else:
        # 사용자 질문 -> 첫 번째 -> 재구성된 질문 = 사용자 질문
        query = latest_question

    print(f"[결과] 재구성된 쿼리: {query}")

    # 답변 생성(return) -> dict 자료형 -> LangGraph 시스템 -> AgentState 자료형 변환 -> 상태 업데이트
    return {"rephrased_query": query}

In [13]:
# 2. 검색 노드 함수 정의 (비동기 처리 방식)
async def retriever_node(state: AgentState) -> dict:
    print("에이전트 2: 관련 매뉴얼 검색")

    # 상태 -> 재구성된 질문 추출
    query = state.rephrased_query

    # 검색어: 재구성된 질문 -> 검색 도구 비동기 처리 방식 실행 -> 결과: [Document]
    # 비동기 처리 방식 실행 -> invoke() 함수 -> await, ainvoke() 함수
    retrieved_docs = await retriever.ainvoke(query)

    # 검색된 문서 리스트를 순회하며 텍스트(page_content) 추출
    contents=[]
    for doc in retrieved_docs:
        text=doc.page_content
        contents.append(text)

    # 리스트(contents)의 모든 문자열 -> join 함수 -> "\n\n" 기준으로 연결
    context_text = "\n\n".join(contents)
    print(f'답변의 근거: \n{context_text}')
    print('-'*80)

    # 함수 실행의 결과(return) -> dict 자료형 -> LangGraph 시스템 -> AgentState 자료형 변환 -> 상태 업데이트
    return {"context": context_text}

In [14]:
# 3. 답변 생성 노드 정의 (비동기 처리 방식)
async def answer_node(state: AgentState) -> dict:
    print("에이전트 3: 최종 답변 작성")
    print('-'*80)

    # 대화 스토리 -> 사용자 최신 질문 추출 -> HumanMessage=state.messages[-1]
    latest_question = state.messages[-1].content 

    # 대화 스토리 -> 사용자 최신 질문 이전 대화 내역 추출
    chat_history = state.messages[:-1]

    # 상태 -> context 추출(답변의 근거)
    context = state.context

    # qa_chain 호출 -> 비동기 처리 방식 -> 답변 생성(AIMessage)
    # 비동기 처리 -> invoke() -> await, ainvoke() 
    response = await qa_chain.ainvoke({
        "context": context,
        "chat_history": chat_history,
        "input": latest_question
    })

    # 함수 실행 결과(return) -> dict 자료형 -> LangGraph 시스템 -> AgentState 자료형 변환 -> 상태 업데이트
    return {"messages": [response]} 

## 그래프 구성 및 컴파일

In [ ]:
# 비동기 처리 방식 노드 함수 -> 그래프 -> 노드 -> compile -> mult_agent

## chatbot agent 실행

In [ ]:
# main 함수 정의
async def main():

    # while 반복문 작성
    while True:

        try:

            # agent 실행 -> 비동기 처리 방식 -> invoke() -> awit, ainvoke()  
            final_state = await multi_agent.ainvoke()



        except Exception as e:
            print(f'오류가 발생했습니다.: {e}')

# main 함수: 비동기 방식 실행 -> jupyter notebook 실행
# main 함수: 비동기 방식 실행 -> python 파일 -> asyncio.run(main())
await main()